In [1]:
import psycopg
from ingest import load_faq_data 
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from rag_helper import RAGPgVector, RAGBase, RAGVector

In [2]:
# Create a connection to the database
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="faq",
    user="user",
    password="pswd",
)

# Enable pgvector extension for storing/searching embedding vectors
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=user database=faq) at 0x120c0b650>

In [3]:
# Load the environment variables
load_dotenv()
# Create an OpenAI client
openai_client = OpenAI()
# Create an embedder
embedder = SentenceTransformer('all-MiniLM-L6-v2')
# Create a connection to the database
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="faq",
    user="user",
    password="pswd",
)
# Enable pgvector extension for storing/searching embedding vectors
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost user=user database=faq) at 0x120cec650>

In [4]:
vector_assistant = RAGPgVector(
    embedder=embedder,
    conn=conn,
    llm_client=openai_client)

In [5]:
vector_assistant.rag_answer("the program has already begun, can I still sign up?")

'Yes — you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

# Summary
## Which search tool should I use?

We built the same RAG assistant in three ways.  

**Same idea each time:** Turn the user’s question into a vector (embedding) → search the stored FAQs most similar to the user question → send matched FAQs text to the LLM.  

The only difference is **where you store and search those vectors**.  

Same RAG flow every time:
1. Embed the user question
2. Find similar FAQ documents
3. Send those documents to the LLM as context

Only the **storage/search backend** changes.

---

## 1. minsearch
- Runs in Python memory
- No database setup
- Data is lost when the notebook/kernel stops
- **Use when:** learning, quick tests, course notebooks

| Good for | Not good for |
|----------|--------------|
| Learning | Real apps with many users |
| Quick experiments in a notebook | Saving data after you close the notebook |

**Think of it as:** sticky notes on your desk: fast, simple, gone when you're done.

---

## 2. SQLite search
- Stores vectors in a local `.db` file
- No separate server needed
- Data persists after restart
- Works for one user / low traffic
- **Use when:** small side projects, demos, portfolio apps

| Good for | Not good for |
|----------|--------------|
| Personal projects | Many people hitting it at once |
| Data that survives restarts | Huge scale (millions of records) |

**Think of it as:** a notebook you keep on the shelf that is yours and small.

---

## 3. PGVector
- Stores vectors in PostgreSQL
- Needs Docker + Postgres setup
- Handles many users and large datasets
- Supports transactions and concurrent access
- **Use when:** production apps, team systems, existing Postgres apps

| Good for | Not good for |
|----------|--------------|
| Apps with real users | “I just want to try RAG in 5 minutes” |
| Millions of documents | Solo notebook experiments |
| Teams already using Postgres | |

**Think of it as:** a library with a catalog system: more setup, but built for many people and lots of data.

---

## Decision rule

| Situation | Tool |
|-----------|------|
| Learning / experimenting | minsearch |
| Small personal app, data must persist | SQLite |
| Real app with users / scale / Postgres already in use | PGVector |

## One sentence to remember
> - **Notebook experiment** → *minsearch*
> - **Small personal project** → *SQLite*
> - **Real product** → *PGVector*
